<a href="https://colab.research.google.com/github/hazirathabasum02/Readme.md/blob/main/PRO4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

INSTALL LIBRARIES

In [ ]:
!pip install -q gymnasium[classic-control] stable-baselines3[extra] imageio[ffmpeg] pyvirtualdisplay

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 7.6 MB/s eta 0:00:00


START VIRTUAL DISPLAY

In [ ]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

CREATE ENVIRINMENT AND TEST

In [ ]:
import gymnasium as gym
from gymnasium import RewardWrapper
from stable_baselines3.common.monitor import Monitor

ENV_ID = "MountainCarContinuous-v0"

class RewardForwardBoost(RewardWrapper):
    """Give extra reward when moving toward the flag"""
    def reward(self, reward):
        position = self.env.state[0]      # car X position
        return reward + 0.3 * position    # small bonus for moving right

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


TRAIN THE MODEL

In [ ]:
# Robust training cell (copy-paste into Colab)
import traceback
import os
import gymnasium as gym
from gymnasium import RewardWrapper
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
import numpy as np

ENV_ID = "MountainCarContinuous-v0"
MODEL_NAME = "ppo_mountaincar"
VIDEO_FOLDER = "logs/videos/"
os.makedirs(VIDEO_FOLDER, exist_ok=True)

# --- Reward wrapper: small forward bonus to encourage climbing ---
class RewardForwardBoost(RewardWrapper):
    """
    Adds small bonus proportional to horizontal position (positive to the right).
    For Gymnasium envs, env.unwrapped.state or env.state may be available.
    """
    def reward(self, reward):
        # Try to get position robustly
        pos = 0.0
        try:
            # Many MountainCar envs store [pos, vel] in env.state or env.unwrapped.state
            if hasattr(self.env, "state"):
                pos = float(self.env.state[0])
            elif hasattr(self.env.unwrapped, "state"):
                pos = float(self.env.unwrapped.state[0])
            else:
                # fallback: read observation if available (not ideal during wrapper call)
                pos = 0.0
        except Exception:
            pos = 0.0
        # small bonus toward the right
        return reward + 0.25 * pos

# --- env factory for vectorized envs (safe Monitor + wrapper) ---
def make_env_fn():
    def _init():
        env = gym.make(ENV_ID)               # no render_mode for training env
        env = RewardForwardBoost(env)        # apply reward shaping
        env = Monitor(env)                   # required by SB3 callbacks/logging
        return env
    return _init

# Training hyperparams (conservative but effective)
N_ENVS = 1           # use 1 in Colab for stability; increase if you know Subproc works
TOTAL_TIMESTEPS = 300_000
EVAL_FREQ = 10_000
N_EVAL_EPISODES = 5

# PPO policy/network
policy_kwargs = dict(
    net_arch=[dict(pi=[256, 256], vf=[256, 256])]
)

try:
    # Create a vectorized env. Try Subproc first if you want parallelism, otherwise Dummy.
    if N_ENVS > 1:
        try:
            vec_env = SubprocVecEnv([make_env_fn() for _ in range(N_ENVS)])
            print("Using SubprocVecEnv with", N_ENVS, "envs")
        except Exception as e_sub:
            print("SubprocVecEnv failed, falling back to DummyVecEnv:", str(e_sub))
            vec_env = DummyVecEnv([make_env_fn() for _ in range(N_ENVS)])
    else:
        vec_env = DummyVecEnv([make_env_fn() for _ in range(N_ENVS)])
        print("Using DummyVecEnv with", N_ENVS, "env(s)")

    # Single eval env (no vectorization) for EvalCallback
    eval_env = Monitor(gym.make(ENV_ID))

    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path="./logs/best_model/",
        log_path="./logs/results/",
        eval_freq=EVAL_FREQ,
        n_eval_episodes=N_EVAL_EPISODES,
        deterministic=True,
    )

    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        tensorboard_log="./ppo_mountaincar_tb/",
        policy_kwargs=policy_kwargs,
        learning_rate=3e-4,
        n_steps=2048,
        batch_size=256,
        n_epochs=10,
        gamma=0.999,
    )

    # Train
    print("Starting training for", TOTAL_TIMESTEPS, "timesteps...")
    model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback)
    model.save(MODEL_NAME)
    print("Training finished and model saved as:", MODEL_NAME)

    # cleanup
    vec_env.close()
    eval_env.close()

except Exception as e:
    print("An error occurred during training. Full traceback below:\n")
    traceback.print_exc()
    print("\nIf this error persists, please copy-paste the above traceback here and I'll diagnose it.")

Using DummyVecEnv with 1 env(s)
Using cpu device


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


Starting training for 300000 timesteps...
Logging to ./ppo_mountaincar_tb/PPO_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 999      |
|    ep_rew_mean     | -181     |
| time/              |          |
|    fps             | 1334     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 999          |
|    ep_rew_mean          | -180         |
| time/                   |              |
|    fps                  | 1009         |
|    iterations           | 2            |
|    time_elapsed         | 4            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0017010127 |
|    clip_fraction        | 0.000586     |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.41        |
|    explained_variance   | 0.032        |
|    learning_r

RECORD VIDEO



In [ ]:
import gymnasium as gym
import imageio
from stable_baselines3 import PPO

env = gym.make(ENV_ID, render_mode="rgb_array")
model = PPO.load("ppo_mountaincar")

frames = []
obs, info = env.reset()
done = False
truncated = False
steps = 0

while not (done or truncated) and steps < 1000:
    frame = env.render()
    frames.append(frame)
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = env.step(action)
    steps += 1

env.close()

video_path = "mountaincar_continuous_video.mp4"
imageio.mimsave(video_path, frames, fps=30)
print("🎥 Video saved to:", video_path)

/usr/local/lib/python3.12/dist-packages/pygame/pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/usr/local/lib/python3.12/dist-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google.cloud')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-pa

🎥 Video saved to: mountaincar_continuous_video.mp4


DOWNLOAD VIDEO


In [ ]:
from google.colab import files
files.download("mountaincar_continuous_video.mp4")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>